# Q2: Iterative Policy Evaluation – 4×4 Grid World

**Silver Lecture 3, pages 10–11**

This notebook:
1. Shows hand-computed V₁, V₂, V₃ compared with code output
2. Runs for any k
3. Lets k → ∞ and analyzes convergence

In [ ]:
import sys

sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np

from src.q2_grid_world.grid_world import (
    N_STATES,
    TERMINAL_STATES,
    display_grid,
    display_policy,
    greedy_policy,
    iterative_policy_eval,
    run_to_convergence,
)

## Grid World Setup

```
 T   1   2   3     (T = terminal, V = 0)
 4   5   6   7
 8   9  10  11
12  13  14   T
```

- Random policy: 1/4 probability each direction (N, S, W, E)
- Reward = −1 per step (except terminal)
- γ = 1 (undiscounted)

## k = 1: First Sweep

All V₁(s) = −1.0 for non-terminal s (since V₀=0 everywhere).

In [ ]:
V1 = iterative_policy_eval(k=1)
display_grid(V1, title='V_1 (k=1)')
print('\nExpected: all non-terminal = -1.0')

## k = 2: Second Sweep

Hand computation for state 1 (top-row, col-1):
- N → 1 (wall reflect): −1 + (−1) = −2
- S → 5: −1 + (−1) = −2  
- W → 0 (terminal): −1 + 0 = **−1**
- E → 2: −1 + (−1) = −2

V₂(1) = (1/4)(−2 − 2 − 1 − 2) = **−1.75**

In [ ]:
V2 = iterative_policy_eval(k=2)
display_grid(V2, title='V_2 (k=2)')
print(f'\nV_2(state 1) = {V2[1]:.4f}  (expected: -1.75)')
print(f'V_2(state 5) = {V2[5]:.4f}  (expected: -2.00)')

## k = 3: Third Sweep

In [ ]:
V3 = iterative_policy_eval(k=3)
display_grid(V3, title='V_3 (k=3)')

# Hand check for state 1
# N->1: -1+V2(1)=-2.75; S->5: -1+(-2)=-3; W->0: -1+0=-1; E->2: -1+(-2)=-3
hand_v3_1 = 0.25 * (-2.75 + (-3) + (-1) + (-3))
print(f'\nHand-computed V_3(state 1) = {hand_v3_1:.4f}')
print(f'Code result   V_3(state 1) = {V3[1]:.4f}')

## Any k

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
ks = [1, 2, 3, 10, 50, 100]

all_vals = np.concatenate([iterative_policy_eval(k) for k in ks])
vmin, vmax = all_vals.min(), 0.0

for ax, k in zip(axes.flat, ks):
    Vk = iterative_policy_eval(k).reshape(4, 4)
    im = ax.imshow(Vk, cmap='RdYlGn', vmin=vmin, vmax=vmax)
    for r in range(4):
        for c in range(4):
            s = r * 4 + c
            txt = 'T' if s in TERMINAL_STATES else f'{Vk[r,c]:.1f}'
            ax.text(c, r, txt, ha='center', va='center', fontsize=9, fontweight='bold')
    ax.set_title(f'k = {k}', fontsize=12)
    ax.set_xticks([]); ax.set_yticks([])
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle('Iterative Policy Evaluation – 4×4 Grid World', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('q2_grid_iterations.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: q2_grid_iterations.png')

## k → ∞: Convergence Analysis

In [ ]:
V_conv, k_conv = run_to_convergence(theta=1e-6)
print(f'Converged after {k_conv} sweeps')
display_grid(V_conv, title=f'V_∞ (converged at k={k_conv})')

In [ ]:
# Track convergence
V = np.zeros(N_STATES)
V_prev = V.copy()
deltas = []

for k in range(1, 400):
    from src.q2_grid_world.grid_world import iterative_policy_eval_step
    V = iterative_policy_eval_step(V)
    deltas.append(float(np.max(np.abs(V - V_prev))))
    V_prev = V.copy()
    if deltas[-1] < 1e-6:
        break

plt.figure(figsize=(8, 4))
plt.plot(deltas, lw=2, color='steelblue')
plt.xlabel('Sweep k')
plt.ylabel('max|V_{k+1} - V_k|')
plt.title('Convergence of Iterative Policy Evaluation')
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('q2_convergence.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: q2_convergence.png')
print('\nComment: The values decrease monotonically and convergence is linear (episodic MDP).')
print('At k→∞, V^π represents the expected total undiscounted return under the random walk policy.')
print('Cells near terminals converge to values near 0; interior cells converge to large negative values.')

In [ ]:
# Show greedy policy from V_inf
pi_conv = greedy_policy(V_conv)
display_policy(pi_conv, title='Greedy Policy from V_∞ (arrows → terminal)')